# 07. LightTools Integration — Complete Guide

## LightTools 시뮬 데이터의 두 가지 역할
1. **학습용**: L_I target (센서면 강도 가이드)
2. **검증용**: 학습 완료 후 PINN PSF vs LT PSF 비교

## 왜 위상 없어도 L_I에 쓸 수 있는가?
```
L_I는 강도(intensity)끼리 비교:
  L_I = mean( ( |U_PINN(z=0)|^2 - I_LT(z=0) )^2 )
              PINN 강도 출력     LT 강도 출력

→ 둘 다 강도이므로 위상 정보 불필요
→ LT ray tracing 결과를 학습 가이드로 사용 가능
```

## 이 노트북의 8단계

| Step | 내용 |
|------|------|
| 1 | LT 모델 전체 구조 구축 (처음부터) |
| 2 | AR Thin Film 코팅 설정 (★ 핵심) |
| 3 | Light Source 설정 (입사각 매핑) |
| 4 | Receiver 설정 (OPD + 검증용 z=40) |
| 5 | z=40 강도 일관성 검증 (TMM+ASM vs LT) |
| 6 | 시뮬레이션 계획 (LHS 샘플링) |
| 7 | 배치 시뮬 실행 |
| 8 | L_I target → PINN 학습 연결 |

In [ ]:
import sys, json
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
print(f'Project: {PROJECT_ROOT.name}')

---
# Step 1: LightTools Model — 전체 구조 구축

## 1.1 새 모델 생성
```
LightTools → File → New
  Units: Micrometers (um)
  Wavelength: 520 nm
```

## 1.2 구조물 생성 순서 (아래에서 위로)
```
┌───────────────────────────────────────────────────────────────────┐
│                  LightTools 모델 구축 순서                          │
├───────────────────────────────────────────────────────────────────┤
│                                                                   │
│  (1) OPD Receiver (z=0)                                           │
│      Insert → Receiver → Planar                                   │
│      Position z=0, Size x=504 um, Resolution 1000 pixels          │
│                                                                   │
│  (2) Encapsulation (z=0~20)                                       │
│      Insert → Solid → Block                                       │
│      504 x 20 um, Material n=1.52, bottom at z=0                  │
│                                                                   │
│  (3) BM1 Aperture (z=20)                                          │
│      7개 슬릿이 뚫린 불투명 판                                      │
│      → Opaque plate + 7 rectangular holes (pitch=72um)            │
│      → 또는 7개 Aperture Stop 배치                                 │
│      ★ slit width = w1, offset = delta_bm1 (시뮬마다 변경)        │
│      ★ Absorbing (반사 X, 투과 X)                                  │
│                                                                   │
│  (4) ILD (z=20~40)                                                │
│      Block 504 x 20 um, n=1.52, bottom at z=20                    │
│                                                                   │
│  (5) BM2 Aperture (z=40)                                          │
│      BM1과 동일 구조, z=40, slit width=w2, offset=delta_bm2       │
│                                                                   │
│  (6) Cover Glass (z=40~590)                                       │
│      Block 504 x 550 um, Glass n=1.52, bottom at z=40             │
│      ★ 윗면(z=590)에 AR Thin Film 적용 → Step 2                   │
│                                                                   │
│  (7) Light Source (z > 590)                                       │
│      → Step 3에서 상세 설명                                        │
└───────────────────────────────────────────────────────────────────┘
```

## 1.3 BM 상세 설정
```
BM = 7개 슬릿이 뚫린 불투명 판

  각 슬릿 중심: x = i*72 + 36 + delta  (i = 0, 1, ..., 6)
  슬릿 폭: w
  BM 속성: Absorbing (Reflectance=0, Transmittance=0)

  LightTools 구현 방법:
    Option A: 큰 Opaque plate에 Boolean subtract로 7개 구멍
    Option B: Aperture Array 기능
    Option C: 7개 개별 Rectangular Aperture를 72um 간격 배치
```

In [ ]:
# ── 만들어야 할 구조 미리 확인 ──
from notebooks.helpers.visualization import plot_pinn_domain

fig, ax = plot_pinn_domain(delta_bm1=0, delta_bm2=0, w1=10, w2=10, figsize=(14, 6))
ax.set_title('LightTools Model Layout (default: delta=0, w=10)', fontsize=12, fontweight='bold')
plt.show()
print('Black = BM (Opaque), White = Slit (Light passes), Purple = OPD Receiver')

---
# Step 2: AR Thin Film Coating 설정 (★ 핵심)

## 왜 Thin Film으로 설정해야 하는가?
```
LT Thin Film 모듈 내부 = Transfer Matrix Method (TMM)
우리 tmm_calculator.py  = Transfer Matrix Method (TMM)
→ 같은 레이어 구성 → 같은 투과 강도 → 일관성 보장!

❌ Simple Reflectance로 설정하면?
   → 고정 반사율만 적용, 각도 의존성 없음
   → 우리 TMM과 불일치 → L_I 데이터 무효

✅ Thin Film Stack으로 설정하면?
   → LT가 자체 TMM으로 각도별 T(theta) 자동 계산
   → 우리 TMM과 일치 → L_I 데이터 유효
```

## 2.1 설정 GUI 순서
```
1. Cover Glass 윗면(z=590) Surface 선택
   → Object Tree > Cover_Glass > Top Surface

2. Properties → Coating 탭
   → Type: "Thin Film Stack" 선택 (Simple가 아님!)

3. Thin Film Editor 열기 → Layer 추가:

   ┌─────────────────────────────────────────────────┐
   │  Thin Film Stack Editor                          │
   │                                                  │
   │  Incident Medium:  Air  (n = 1.00)              │
   │                                                  │
   │  Layer 1:  SiO2   n = 1.46   d = 34.6 nm       │
   │  Layer 2:  TiO2   n = 2.35   d = 25.9 nm       │
   │  Layer 3:  SiO2   n = 1.46   d = 20.7 nm       │
   │  Layer 4:  TiO2   n = 2.35   d = 169.5 nm      │
   │                                                  │
   │  Substrate: Glass (n = 1.52)                    │
   │                                                  │
   │  ★ 단위 주의: LT가 um 단위면 nm→um 변환!       │
   │    34.6 nm = 0.0346 um                          │
   │    25.9 nm = 0.0259 um                          │
   │    20.7 nm = 0.0207 um                          │
   │   169.5 nm = 0.1695 um                          │
   └─────────────────────────────────────────────────┘

4. Wavelength: 520 nm 확인
5. Apply
```

## 2.2 설정 검증

LT Thin Film Editor에서 **Transmittance vs Angle** 그래프를 표시하고,
아래 TMM 계산 결과와 비교하세요:

In [ ]:
# ── TMM 계산: LT Thin Film Editor 곡선과 비교용 ──
from backend.physics.tmm_calculator import GorillaDXTMM

tmm = GorillaDXTMM()
theta_arr = np.arange(0, 42, 1.0)
lut = tmm.compute_lut(theta_arr)

# Power transmittance: T = |t|^2 * n_glass / n_air
T_power = lut['t_amplitude']**2 * 1.52
R_power = 1 - T_power

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(theta_arr, T_power * 100, 'b-', lw=2)
ax.set_xlabel('Incidence Angle (deg)', fontsize=11)
ax.set_ylabel('Transmittance T (%)', fontsize=11)
ax.set_title('AR Transmittance\n(LT Thin Film Editor graph must match this!)', fontsize=11)
ax.set_ylim([90, 102])
ax.grid(True, alpha=0.3)
ax.plot(0, T_power[0]*100, 'ro', ms=10)
ax.annotate(f'T(0) = {T_power[0]*100:.1f}%', xy=(1, T_power[0]*100-2),
            fontsize=12, color='red', fontweight='bold')
ax.plot(30, T_power[30]*100, 'ro', ms=10)
ax.annotate(f'T(30) = {T_power[30]*100:.1f}%', xy=(31, T_power[30]*100-2),
            fontsize=12, color='red', fontweight='bold')

ax = axes[1]
ax.plot(theta_arr, R_power * 100, 'r-', lw=2)
ax.set_xlabel('Incidence Angle (deg)', fontsize=11)
ax.set_ylabel('Reflectance R (%)', fontsize=11)
ax.set_title('AR Reflectance', fontsize=11)
ax.grid(True, alpha=0.3)
ax.plot(0, R_power[0]*100, 'ro', ms=10)
ax.annotate(f'R(0) = {R_power[0]*100:.2f}%', xy=(1, R_power[0]*100+0.5),
            fontsize=12, color='red', fontweight='bold')

plt.suptitle('TMM Calculation — LT Thin Film Editor에서 같은 곡선이 나와야 함',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== LT 검증 체크포인트 ===')
print(f'  T(0 deg)  = {T_power[0]*100:.2f}%  ← LT 값과 비교')
print(f'  T(30 deg) = {T_power[30]*100:.2f}%  ← LT 값과 비교')
print(f'  R(0 deg)  = {R_power[0]*100:.2f}%')
print(f'  차이 < 0.5%이면 OK')

---
# Step 3: Light Source 설정

## 3.1 Source 생성
```
Insert → Source → Surface Source (Planar)

  ┌─────────────────────────────────────────┐
  │  Source Properties                       │
  │                                          │
  │  Type: Planar (평면 광원)                │
  │  Wavelength: 520 nm                     │
  │  Power: 1.0 W                           │
  │  Size X: 504 um (도메인 전체)            │
  │  Position: z > 590 um (AR 위)           │
  │  Angular: Collimated (평행광)            │
  │  Tilt X: theta deg  ★ 시뮬마다 변경     │
  │  Ray Count: >= 100,000                  │
  └─────────────────────────────────────────┘
```

## 3.2 우리 파이프라인과 각도 매핑
```
우리 theta (degrees)  =  LT Source Tilt X (degrees)
→ 1:1 매핑 (같은 값 넣으면 됨)

  theta = 0   → Tilt X = 0    (수직 입사)
  theta = 15  → Tilt X = 15
  theta = 30  → Tilt X = 30
  theta = 40  → Tilt X = 40   (최대각)

  theta > 41.1 → 전반사 (시뮬 범위 밖)
  음수 각도 → 대칭이므로 양수만 시뮬
```

## 3.3 입사광이 PINN과 일치하는 원리
```
Light Source (평면파, theta도)
    ↓
AR Thin Film (LT 내부 TMM으로 T(theta) 계산)
    ↓  ← 우리 TMM과 동일 결과 (같은 레이어, 같은 방법)
Cover Glass 550um (LT ray tracing)
    ↓  ← 균일 매질 전파는 ray tracing = ASM 동일
z=40 면 도달 (강도 기준)
    ↓  ← |U_ASM|^2 = I_LT 일치! (Step 5에서 검증)
BM2 → ILD → BM1 → Encap → OPD
    ↓
OPD Receiver에서 I(x) 측정 → L_I target
```

In [ ]:
# ── 각도별 z=40 강도 (LT에서 이렇게 보여야 함) ──
from backend.training.loss_functions import ASMIncidentLUT

asm_lut = ASMIncidentLUT(str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'))
x_vis = torch.linspace(0, 504, 500)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, theta in zip(axes, [0, 15, 30, 40]):
    sin_t = np.sin(np.radians(theta))
    U_re, U_im = asm_lut.lookup(x_vis, torch.full((500,), sin_t))
    I = (U_re**2 + U_im**2).numpy()
    ax.plot(x_vis.numpy(), I, 'b-', lw=1)
    ax.set_title(f'theta={theta} deg | I_mean={I.mean():.4f}', fontsize=10)
    ax.set_xlabel('x (um)')
    ax.set_ylabel('Intensity')
    ax.grid(True, alpha=0.3)

plt.suptitle('TMM+ASM Intensity at z=40 (BM 없을 때 LT z=40 Receiver와 일치해야 함)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Step 4: Receiver (검출기) 설정

## 4.1 OPD Receiver (z=0) — L_I 데이터 수집용
```
Insert → Receiver → Planar Receiver

  Name:       "OPD_Receiver"
  Position:   z = 0 um
  Size X:     504 um
  Pixels X:   1000  (dx = 0.504 um)
  Data Type:  Irradiance (W/um^2)

  → 이 Receiver의 1D 강도 프로파일 = L_I target
```

## 4.2 z=40 Verification Receiver — 일관성 검증용 (임시)
```
Insert → Receiver → Planar Receiver

  Name:       "z40_Receiver"
  Position:   z = 40 um (BM2 바로 위, CG 바로 아래)
  Size X:     504 um
  Pixels X:   1000

  ★ Step 5 검증 전용
  ★ 검증 시 BM1, BM2를 비활성화(Disable)하고 측정!
  ★ 검증 완료 후 삭제 또는 비활성화 가능
```

## 4.3 Receiver 데이터 추출 방법
```
시뮬 완료 후:

  GUI 방식:
    1. Receiver 더블클릭 → Irradiance map 표시
    2. Tools → Cross Section → X축 따라 1D 프로파일
    3. File → Export → Tab-separated text
    4. 또는 Data → Copy to Clipboard → Excel에 붙여넣기

  COM API 방식:
    data = lt.GetReceiverData("OPD_Receiver")

  저장 포맷 (numpy):
    np.savez("sim_0001.npz",
        intensity=I_array,     # (N,) float, 1D intensity
        x_coords=x_array,      # (N,) float, x in um
        psf_7=psf_array,       # (7,) float, 7 OPD pixels
        delta_bm1=0.0,         # 설계변수
        delta_bm2=0.0,
        w1=10.0,
        w2=10.0,
        theta_deg=0.0,
    )
```

---
# Step 5: z=40 강도 일관성 검증 (★ 가장 중요)

## 검증 원리
```
비교하는 것: |U_ASM(z=40)|^2  vs  I_LT(z=40)
             우리 TMM+ASM 강도    LT Thin Film + Ray Trace 강도

LT Thin Film = 내부적으로 TMM
LT Ray Trace through CG = 균일 매질 전파 = ASM과 동등
→ 강도 기준 동일 결과 기대
```

## 검증 절차
```
1. LightTools에서 BM1, BM2를 Disable (비활성화)
   → Object Tree > BM1 > 우클릭 > Disable
   → Object Tree > BM2 > 우클릭 > Disable
   (AR + CG만 남은 상태)

2. z40_Receiver가 z=40에 있는지 확인

3. theta=0 으로 시뮬 실행 → z40_Receiver 데이터 추출

4. theta=15, 30도로 반복

5. 결과를 아래 형식으로 저장:
   np.savez("data/lt_results/z40_verify_theta0.npz",
       x_coords=x_array,   # um
       intensity=I_array,   # irradiance
   )

6. 아래 셀 실행 → 파란선(TMM+ASM)과 빨간선(LT) 비교

7. 차이 < 5% → OK! → BM 다시 Enable → 배치 시뮬 진행
   차이 > 5% → AR Thin Film 설정 재확인
```

In [ ]:
# ── z=40 강도 일관성 검증 ──
lt_dir = PROJECT_ROOT / 'data' / 'lt_results'

test_angles = [0, 15, 30]
fig, axes = plt.subplots(1, len(test_angles), figsize=(5*len(test_angles), 4.5))

for ax, theta in zip(axes, test_angles):
    # Our TMM+ASM intensity
    sin_t = np.sin(np.radians(theta))
    U_re, U_im = asm_lut.lookup(x_vis, torch.full((500,), sin_t))
    I_asm = (U_re**2 + U_im**2).numpy()
    ax.plot(x_vis.numpy(), I_asm, 'b-', lw=2, label='TMM+ASM (ours)')
    
    # LT verification data (if exists)
    lt_file = lt_dir / f'z40_verify_theta{theta}.npz'
    if lt_file.exists():
        lt_data = np.load(lt_file)
        ax.plot(lt_data['x_coords'], lt_data['intensity'], 'r--', lw=2, label='LightTools')
        I_lt_interp = np.interp(x_vis.numpy(), lt_data['x_coords'], lt_data['intensity'])
        rel_err = np.abs(I_asm - I_lt_interp).mean() / (I_asm.mean() + 1e-8)
        passed = rel_err < 0.05
        ax.set_title(f'theta={theta} | error={rel_err*100:.2f}% | '
                     f'{"PASS" if passed else "FAIL"}',
                     fontsize=11, color='green' if passed else 'red', fontweight='bold')
    else:
        ax.set_title(f'theta={theta} | Awaiting LT data', fontsize=11, color='gray')
    
    ax.set_xlabel('x (um)')
    ax.set_ylabel('Intensity')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Step 5: z=40 Intensity Consistency (TMM+ASM vs LightTools)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('LT 검증 데이터 저장 코드:')
print('  import numpy as np')
print('  np.savez("data/lt_results/z40_verify_theta0.npz",')
print('      x_coords=x_array,  intensity=I_array)')

---
# Step 6: Simulation Plan (LHS Sampling)

In [ ]:
from backend.data.lhs_sampler import generate_lhs_samples, save_simulation_plan

N_DESIGNS = 40
N_ANGLES = 5
plan = generate_lhs_samples(n_samples=N_DESIGNS, n_angles=N_ANGLES, seed=42)

print(f'Designs: {N_DESIGNS}, Angles: {plan["theta_values"]}, Total: {plan["n_total"]}')

params = plan['design_params']
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, i, name in zip(axes, range(4), plan['param_names']):
    ax.hist(params[:, i], bins=12, color='steelblue', edgecolor='white')
    ax.set_title(name)
    ax.set_xlabel('um')
plt.suptitle('LHS Design Space Coverage', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

plan_path = PROJECT_ROOT / 'data' / 'lt_checkpoint' / 'simulation_plan.json'
save_simulation_plan(plan, str(plan_path))
print(f'Saved: {plan_path}')

---
# Step 7: Batch Simulation

## 시뮬마다 변경할 것
```
변경:  BM1 (w1, delta1), BM2 (w2, delta2), Source angle (theta)
고정:  AR Thin Film, Cover Glass, Wavelength, Receiver
```

In [ ]:
# ── COM API 자동 실행 ──
RUN_LIGHTTOOLS = False  # True로 변경하면 실제 LT 실행

if RUN_LIGHTTOOLS:
    from backend.data.lighttools_runner import LightToolsRunner
    runner = LightToolsRunner(
        model_path=r'C:\LightTools\UDFPS_model.lts',  # ← 실제 경로
        max_retries=3,
    )
    try:
        runner.connect()
        results = runner.run_batch(
            configs=plan['all_configs'],
            output_dir=str(PROJECT_ROOT / 'data' / 'lt_results'),
            checkpoint_every=10,
        )
        print(f'Done: {sum(1 for r in results if r.success)}/{len(results)} OK')
    finally:
        runner.disconnect()
else:
    print('RUN_LIGHTTOOLS = False (비활성화)')
    print()
    print('=== 수동 실행 시: 각 시뮬에서 변경할 값 ===')
    for cfg in plan['all_configs'][:5]:
        print(f'  #{cfg["sim_id"]:3d}: BM1(w={cfg["w1"]:.1f}, d={cfg["delta_bm1"]:+.1f}) '
              f'BM2(w={cfg["w2"]:.1f}, d={cfg["delta_bm2"]:+.1f}) '
              f'Tilt={cfg["theta_deg"]:.0f}')
    print(f'  ... 총 {plan["n_total"]}개')

---
# Step 8: L_I Target → PINN 학습 연결

In [ ]:
# ── 결과 확인 ──
result_files = sorted(lt_dir.glob('sim_*.npz'))

if not result_files:
    print(f'No results in {lt_dir}. Complete Step 7 first.')
    print()
    print('Phase C-1: L_I 없이 학습 가능 (L_H + L_phase + L_BC)')
    print('Phase C-2: LT 데이터 확보 후 L_I 추가 fine-tuning')
else:
    print(f'{len(result_files)} results found')
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, f in zip(axes.flat, result_files[:6]):
        data = np.load(f)
        ax.plot(data['x_coords'], data['intensity'], 'b-', lw=0.8)
        ax.set_title(f'd1={float(data["delta_bm1"]):+.0f} w1={float(data["w1"]):.0f} '
                     f'th={float(data["theta_deg"]):.0f}', fontsize=8)
        ax.set_xlabel('x (um)', fontsize=8)
        ax.grid(True, alpha=0.2)
    plt.suptitle('LT Results (L_I targets)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── PINN 학습에 L_I 연결 방법 ──
print('='*60)
print('L_I를 PINN 학습에 연결하는 방법')
print('='*60)
print()
print('1. configs/phase_c_full_gpu.yaml 수정:')
print('   curriculum:')
print('     lambda_I: 0.3    # 0.0 → 0.3')
print()
print('2. 학습 코드에서:')
print('   from backend.data.lighttools_runner import LTResultDataset')
print('   lt_dataset = LTResultDataset("data/lt_results")')
print()
print('3. L_I loss 계산:')
print('   cfg, x_lt, I_lt = lt_dataset.get_target(idx)')
print('   # PINN으로 같은 설계변수에서 z=0 강도 예측')
print('   U = model(coords)  # z=0')
print('   I_pinn = U[:, 0]**2 + U[:, 1]**2')
print('   L_I = mean((I_pinn - I_target)**2)')
print()
print('타임라인:')
print('  Phase C-1: L_H + L_phase + L_BC (L_I 없이)')
print('  Phase C-2: + L_I (LT 데이터 추가)')
print('  Phase E:   L_I target을 실측 데이터로 교체')

---
# Checklist

| # | 항목 | 완료 |
|---|------|:----:|
| 1 | LT 모델 구축 (AR+CG+BM+OPD) | [ ] |
| 2 | AR Thin Film 설정 (4-layer, not Simple!) | [ ] |
| 2a | T(0) = TMM 결과와 비교 (< 0.5% 차이) | [ ] |
| 3 | Source: Planar, 520nm, Tilt X = theta | [ ] |
| 4 | OPD Receiver (z=0, 504um, 1000px) | [ ] |
| 4a | z40 Receiver (검증용) | [ ] |
| 5 | BM Disable → z=40 강도 비교 (< 5%) | [ ] |
| 5a | BM 다시 Enable | [ ] |
| 6 | LHS 시뮬 계획 생성 | [ ] |
| 7 | 배치 시뮬 200 runs | [ ] |
| 8 | L_I dataset → PINN lambda_I = 0.3 | [ ] |